# Ejercicio 1 — De Bronze a Silver
---

Este es un extracto de lo que llega desde MongoDB a la capa Bronze: la colección customers, con sus arrays anidados, sus nulls sueltos y un par de inconsistencias típicas de un sistema de origen que no valida mucho antes de guardar.  

Con esto, arma la transformación a Silver. Concretamente:
1. Carga los datos (puedes leerlos como JSON directamente, o construir el DataFrame a mano, como prefieras).
2. Pasa a esquema Silver con la convención de sufijos: customer_id, pais_cd (ojo con el "ec" en minúscula), nombre_txt, email_txt, telefonos_arr, creado_ts, activo_flag como booleano real.
3. Saca las compras (purchases) a su propia tabla, customer_purchases_silver, sin perder la referencia a customer_id y pais_cd.
4. Decide qué hacer con cada null que encuentres (full_name, created_at, email, un amount nulo dentro de una compra, un array de phones vacío) y déjalo explícito en el código
5. Particiona el resultado por país y fecha.

### Imports


In [36]:
from src.common.spark_session import get_spark
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window
from datetime import date

In [37]:
spark = get_spark("ejercicio1-bronze-to-silver")
spark

## Paso 1: Carga de Bronze


In [38]:
# Se define el esquema Bronze con StructType/StructField
bronze_schema = T.StructType([
    T.StructField("_id", T.StringType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("full_name", T.StringType(), True),
    T.StructField("contact", T.StructType([
        T.StructField("phones", T.ArrayType(T.StringType()), True),
        T.StructField("email", T.StringType(), True),
    ]), True),
    T.StructField("created_at", T.StringType(), True),
    T.StructField("is_active", T.StringType(), True),
    T.StructField("purchases", T.ArrayType(T.StructType([
        T.StructField("order_id", T.StringType(), True),
        T.StructField("amount", T.DoubleType(), True),
        T.StructField("currency", T.StringType(), True),
    ])), True),
])

In [39]:
# Se crea el DataFrame Bronze a partir del archivo JSON con el esquema definido
bronze_df = spark.read.option("multiline", True).schema(bronze_schema).json("/workspace/data/bronze/customers.json")

In [40]:
# Esquema del DataFrame Bronze
bronze_df.printSchema()

root
 |-- _id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- phones: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- email: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- purchases: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- order_id: string (nullable = true)
 |    |    |-- amount: double (nullable = true)
 |    |    |-- currency: string (nullable = true)



In [41]:
# Visualizacion del DataFrame Bronze
bronze_df.show(truncate=False)

+------------------------+-------+------------+-------------------------------------------------------+--------------------+---------+---------------------------------------------+
|_id                     |country|full_name   |contact                                                |created_at          |is_active|purchases                                    |
+------------------------+-------+------------+-------------------------------------------------------+--------------------+---------+---------------------------------------------+
|60c1f2a1e4b0a1a2b3c4d5e6|GT     |María López |{[+50255512345, +50255598765], maria.lopez@example.com}|2024-03-11T14:22:00Z|true     |[{ORD-001, 129.5, GTQ}, {ORD-002, 45.0, GTQ}]|
|60c1f2a1e4b0a1a2b3c4d5e7|SV     |Carlos Reyes|{[], NULL}                                             |NULL                |1        |[]                                           |
|60c1f2a1e4b0a1a2b3c4d5e8|ec     |Ana Torres  |{[+593987654321], ana.torres@example.com}       

## Paso 2: Esquema Silver

- Se tuvieron en cuenta la convención de sufijos: customer_id, pais_cd, nombre_txt, email_txt, telefonos_arr, creado_ts, activo_flag.

- En el campo pais_cd se normaliza a mayúsculas, "ec" a "EC".

- En el campo creado_ts existen dos formatos de fecha, con Z (2024-03-11T14:22:00Z) y sin zona ni T (2024-03-12 09:15:00), por lo que se consideran ambos formatos.



In [42]:
# Se transforma el DataFrame Bronze a Silver con las transformaciones requeridas
customers_silver = (
    bronze_df
    .withColumnRenamed("_id", "customer_id")
    .withColumn("pais_cd", F.upper(F.trim(F.col("country"))))
    .withColumn("nombre_txt", F.trim(F.col("full_name")))
    .withColumn("email_txt", F.trim(F.col("contact.email")))
    .withColumn("telefonos_arr", F.when(F.size(F.col("contact.phones")) == 0, None).otherwise(F.col("contact.phones")))
    .withColumn("creado_ts", F.coalesce(
        F.to_timestamp(F.col("created_at"), "yyyy-MM-dd'T'HH:mm:ss'Z'"),
        F.to_timestamp(F.col("created_at"), "yyyy-MM-dd HH:mm:ss"),
    ))
    .withColumn("activo_flag", F.lower(F.trim(F.col("is_active"))).isin("true", "1"))
    .select("customer_id", "pais_cd", "nombre_txt",
            "email_txt", "telefonos_arr",
            "creado_ts", "activo_flag")
)

In [43]:
# Esquema del DataFrame Silver
customers_silver.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- pais_cd: string (nullable = true)
 |-- nombre_txt: string (nullable = true)
 |-- email_txt: string (nullable = true)
 |-- telefonos_arr: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- creado_ts: timestamp (nullable = true)
 |-- activo_flag: boolean (nullable = true)



In [44]:
# Visualizacion del DataFrame Silver
customers_silver.show(truncate=False)

+------------------------+-------+------------+-----------------------+----------------------------+-------------------+-----------+
|customer_id             |pais_cd|nombre_txt  |email_txt              |telefonos_arr               |creado_ts          |activo_flag|
+------------------------+-------+------------+-----------------------+----------------------------+-------------------+-----------+
|60c1f2a1e4b0a1a2b3c4d5e6|GT     |María López |maria.lopez@example.com|[+50255512345, +50255598765]|2024-03-11 14:22:00|true       |
|60c1f2a1e4b0a1a2b3c4d5e7|SV     |Carlos Reyes|NULL                   |NULL                        |NULL               |true       |
|60c1f2a1e4b0a1a2b3c4d5e8|EC     |Ana Torres  |ana.torres@example.com |[+593987654321]             |2024-03-12 09:15:00|false      |
|60c1f2a1e4b0a1a2b3c4d5e9|PE     |NULL        |sin_nombre@example.com |[+51987654321]              |2024-03-13 08:00:00|true       |
+------------------------+-------+------------+----------------------

## Paso 3: Customer_purchases_silver

Se saca "purchases" a su propia tabla, sin perder customer_id ni pais_cd.

In [45]:
# Se extraen las compras de los clientes para obtener su propia tabla 
# sin perder la referencia a customer_id y pais_cd, y se renombran las columnas.
customer_purchases_silver = (
    bronze_df
    .withColumnRenamed("_id", "customer_id")
    .withColumn("pais_cd", F.upper(F.trim(F.col("country"))))
    .withColumn("compra", F.explode(F.col("purchases")))
    .select(
        F.col("compra.order_id").alias("order_id_txt"),
        "customer_id",
        "pais_cd",
        F.col("compra.amount").cast("decimal(18, 2)").alias("monto_dec"),
        F.col("compra.currency").alias("moneda_cd")
    )
)

In [46]:
# Esquema del DataFrame Silver de compras
customer_purchases_silver.printSchema()

root
 |-- order_id_txt: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- pais_cd: string (nullable = true)
 |-- monto_dec: decimal(18,2) (nullable = true)
 |-- moneda_cd: string (nullable = true)



In [47]:
# Visualizacion del DataFrame Silver de compras
customer_purchases_silver.show(truncate=False)

+------------+------------------------+-------+---------+---------+
|order_id_txt|customer_id             |pais_cd|monto_dec|moneda_cd|
+------------+------------------------+-------+---------+---------+
|ORD-001     |60c1f2a1e4b0a1a2b3c4d5e6|GT     |129.50   |GTQ      |
|ORD-002     |60c1f2a1e4b0a1a2b3c4d5e6|GT     |45.00    |GTQ      |
|ORD-003     |60c1f2a1e4b0a1a2b3c4d5e8|EC     |88.25    |USD      |
|ORD-004     |60c1f2a1e4b0a1a2b3c4d5e9|PE     |200.00   |PEN      |
|ORD-005     |60c1f2a1e4b0a1a2b3c4d5e9|PE     |NULL     |PEN      |
+------------+------------------------+-------+---------+---------+



## Paso 4: Particionamiento

- Para la partición de customer_silver por país y fecha, se crea una nueva columna llamada `creado_dt` la cual deriva de `creado_ts`, para evitar problemas con valores nulos, se utiliza Coalesce para asignar un valor centinela de fecha (1900-01-01) en caso de que `creado_ts` sea nulo. 

- Se exportan los DataFrames Silver a formato Delta para su posterior uso.


In [48]:
# Se establecen las rutas de salida para los DataFrames Silver
RUTA_CUSTOMERS_SILVER = "/workspace/data/silver/customers_silver"
RUTA_PURCHASES_SILVER = "/workspace/data/silver/customer_purchases_silver"

# Se crea una nueva columna "creado_dt" en el DataFrame customers_silver para particionar por fecha
customers_silver_particionado = customers_silver.withColumn(
    "creado_dt", F.coalesce(F.to_date(F.col("creado_ts")), F.lit(date(1900, 1, 1)))
)

# customers_silver particionado por pais y fecha
customers_silver_particionado.write.format("delta").mode("overwrite") \
    .partitionBy("pais_cd", "creado_dt").save(RUTA_CUSTOMERS_SILVER)

# customers_purchases_silver particionado por pais
customer_purchases_silver.write.format("delta").mode("overwrite").partitionBy("pais_cd").save(RUTA_PURCHASES_SILVER)


## Tratamiento de valores nulos

Se mantuvieron la mayoria de los null en silver porque representan ausencia de valor y no necesariamente un error de calidad. Reemplazarlos por valores como "desconocido", "sin_nombre" o 0 implicaría asignarle otro sentido al dato original. Esta capa silver podría conservar una representacion fiel de los datos pero normalizados y estandarizados, mientras que para la capa gold podrían hacerse las debidas transformaciones de los null según quien consuma esta tabla, ya sea data science, data analytics.  
Solo se transformó los nulos de la columna creado_ts para poder hacer la partición por país y fecha correctamente. 

## Verificacion

In [49]:
# Se lee customers_silver desde Delta para verificar que se haya guardado correctamente
print("customers_silver leido desde Delta")
spark.read.format("delta").load(RUTA_CUSTOMERS_SILVER).show(truncate=False)


customers_silver leido desde Delta
+------------------------+-------+------------+-----------------------+----------------------------+-------------------+-----------+----------+
|customer_id             |pais_cd|nombre_txt  |email_txt              |telefonos_arr               |creado_ts          |activo_flag|creado_dt |
+------------------------+-------+------------+-----------------------+----------------------------+-------------------+-----------+----------+
|60c1f2a1e4b0a1a2b3c4d5e6|GT     |María López |maria.lopez@example.com|[+50255512345, +50255598765]|2024-03-11 14:22:00|true       |2024-03-11|
|60c1f2a1e4b0a1a2b3c4d5e8|EC     |Ana Torres  |ana.torres@example.com |[+593987654321]             |2024-03-12 09:15:00|false      |2024-03-12|
|60c1f2a1e4b0a1a2b3c4d5e9|PE     |NULL        |sin_nombre@example.com |[+51987654321]              |2024-03-13 08:00:00|true       |2024-03-13|
|60c1f2a1e4b0a1a2b3c4d5e7|SV     |Carlos Reyes|NULL                   |NULL                        |N

In [50]:
# Se lee customer_purchases_silver desde Delta para verificar que se haya guardado correctamente
print("customer_purchases_silver leido desde Delta")
spark.read.format("delta").load(RUTA_PURCHASES_SILVER).show(truncate=False)

customer_purchases_silver leido desde Delta
+------------+------------------------+-------+---------+---------+
|order_id_txt|customer_id             |pais_cd|monto_dec|moneda_cd|
+------------+------------------------+-------+---------+---------+
|ORD-004     |60c1f2a1e4b0a1a2b3c4d5e9|PE     |200.00   |PEN      |
|ORD-005     |60c1f2a1e4b0a1a2b3c4d5e9|PE     |NULL     |PEN      |
|ORD-001     |60c1f2a1e4b0a1a2b3c4d5e6|GT     |129.50   |GTQ      |
|ORD-002     |60c1f2a1e4b0a1a2b3c4d5e6|GT     |45.00    |GTQ      |
|ORD-003     |60c1f2a1e4b0a1a2b3c4d5e8|EC     |88.25    |USD      |
+------------+------------------------+-------+---------+---------+

